In [1]:
from sage.combinat.designs.incidence_structures import IncidenceStructure
from sage.groups.perm_gps.permgroup_named import SymmetricGroup
import time

def binary_matrix_to_blocks(matrix):
    print("Converting binary matrix to blocks...")
    blocks = []
    for row in matrix:
        block = [i for i, x in enumerate(row) if x == 1]
        blocks.append(block)
    return blocks

def hyp_automorphism(incidence_matrix):
    print("Computing the automorphism group...")
    num_vertices = len(incidence_matrix[0])
    print("Number of vertices:", num_vertices)
    points = list(range(num_vertices))
    blocks = binary_matrix_to_blocks(incidence_matrix)
    for j in range(num_vertices):
        if all(row[j] == 0 for row in incidence_matrix):
            blocks.append([j])
    print("Blocks computed:", blocks)
    incidence_structure = IncidenceStructure(points, blocks)
    print("Incidence structure created:")
    aut_g = incidence_structure.automorphism_group()
    print("Automorphism group calculated.")
    return aut_g

def S_n_generators(aut_g):
    print("Generating symmetric group generators...")
    num_vertices = aut_g.degree()
    generators = []
    for perm in aut_g.gens():
        perm_cycles = perm.cycle_tuples()
        perm_string = ""
        for cycle in perm_cycles:
            if len(cycle) > 1:
                adjusted_cycle = [i + 1 for i in cycle]
                perm_string += "(" + " ".join(map(str, adjusted_cycle)) + ")"
        if perm_string:
            generators.append(perm_string)
    print("Generated permutations in cycle form:", generators)  # Debugging print to verify correctness
    return generators

def vertex_orders(incidence_matrix):
    print("Calculating vertex orders...")
    vertex_order_dict = {}
    for j in range(len(incidence_matrix[0])):
        order = sum(row[j] for row in incidence_matrix)
        vertex_order_dict[j + 1] = order
    return vertex_order_dict

def edge_orders(incidence_matrix):
    print("Calculating edge orders...")
    edge_order_dict = {}
    for i, row in enumerate(incidence_matrix):
        order = sum(row)
        edge_order_dict[i + 1] = order
    return edge_order_dict

print("Starting script...")

incidence_matrix = [[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1]]

print("Defined incidence matrix.")

start_time = time.time()
automorphisms = hyp_automorphism(incidence_matrix)
end_time = time.time()

print(f"Automorphism group computation took {end_time - start_time:.2f} seconds.")

if automorphisms is not None:
    if automorphisms.order() == 1:
        print("The automorphism group is trivial.")
    else:
        print("Automorphism group is non-trivial.")
        print("Automorphism group generators are:")
        symmetric_generators = S_n_generators(automorphisms)
        for gen in symmetric_generators:
            print(gen)

        print("Cardinality of aut:", automorphisms.cardinality())

        vertex_order_dict = vertex_orders(incidence_matrix)
        print("Vertex orders:")
        for vertex, order in vertex_order_dict.items():
            print(f"Vertex {vertex} has degree {order}")

        edge_order_dict = edge_orders(incidence_matrix)
        print("Edge orders:")
        for edge, order in edge_order_dict.items():
            vertices = [i + 1 for i, x in enumerate(incidence_matrix[edge - 1]) if x == 1]
            print(f"Edge e{edge} degree {order} {vertices}")

        vertex_edge_dict = {i + 1: [] for i in range(len(incidence_matrix[0]))}
        for i, row in enumerate(incidence_matrix):
            for j, val in enumerate(row):
                if val == 1:
                    vertex_edge_dict[j + 1].append(i + 1)
        print("Vertices and their edges:")
        for vertex, edges in vertex_edge_dict.items():
            print(f"Vertex {vertex} has degree {len(edges)} {['e' + str(e) for e in edges]}")

        from sage.interfaces.gap import gap

        gap.eval('LoadPackage("sonata");')
        gap.eval('LoadPackage("grape");')

        def group_structure_description(generators, expected_cardinality):
            # Properly format each generator for GAP as a list of strings
            gap_generators = [f"PermList([{','.join(gen.replace('(', '').replace(')', '').split())}])" for gen in generators]
            gap_generators_str = ', '.join(gap_generators)
            gap_command = f"Group({gap_generators_str});"
            gap_group = gap.eval(gap_command)
            actual_cardinality = gap.Size(gap_group)

            if actual_cardinality != expected_cardinality:
                print(f"Warning: The actual cardinality {actual_cardinality} does not match the expected cardinality {expected_cardinality}")
                subgroups = gap.Subgroups(gap_group)
                return [gap.Size(sg) for sg in subgroups if gap.Size(sg) == expected_cardinality]
            else:
                print(f"Cardinality matches expected value of {expected_cardinality}")

            structure_description = gap.StructureDescription(gap_group)
            normal_subgroups = gap.NormalSubgroups(gap_group)
            
            return structure_description, normal_subgroups

        group_info = group_structure_description(symmetric_generators, automorphisms.cardinality())
        
        if isinstance(group_info, list):
            print("Subgroups with matching cardinality:", group_info)
        else:
            structure_description, normal_subgroups = group_info
            print(f"Group Structure: {structure_description}")
            print("Normal Subgroups:")
            for nsg in normal_subgroups:
                print(f"  {gap.StructureDescription(nsg)}, Cardinality: {gap.Size(nsg)}")
            

else:
    print("Automorphism group could not be computed.")

print("Script finished.")

Starting script...
Defined incidence matrix.
Computing the automorphism group...
Number of vertices: 12
Converting binary matrix to blocks...
Blocks computed: [[0, 1], [1, 2], [2, 3], [0, 3], [4, 5], [5, 6], [6, 7], [4, 7], [8, 9], [9, 10], [10, 11], [8, 11]]
Incidence structure created:
Automorphism group calculated.
Automorphism group computation took 0.06 seconds.
Automorphism group is non-trivial.
Automorphism group generators are:
Generating symmetric group generators...
Generated permutations in cycle form: ['(10 12)', '(9 10)(11 12)', '(6 8)', '(5 6)(7 8)', '(5 9)(6 10)(7 11)(8 12)', '(2 4)', '(1 2)(3 4)', '(1 5)(2 6)(3 7)(4 8)']
(10 12)
(9 10)(11 12)
(6 8)
(5 6)(7 8)
(5 9)(6 10)(7 11)(8 12)
(2 4)
(1 2)(3 4)
(1 5)(2 6)(3 7)(4 8)
Cardinality of aut: 3072
Calculating vertex orders...
Vertex orders:
Vertex 1 has degree 2
Vertex 2 has degree 2
Vertex 3 has degree 2
Vertex 4 has degree 2
Vertex 5 has degree 2
Vertex 6 has degree 2
Vertex 7 has degree 2
Vertex 8 has degree 2
Vertex 9 

TypeError: error evaluating "Group(PermList([10,12]), PermList([9,1011,12]), PermList([6,8]), PermList([5,67,8]), PermList([5,96,107,118,12]), PermList([2,4]), PermList([1,23,4]), PermList([1,52,63,74,8]));":
argument of type 'RuntimeError' is not iterable

Primal graph adjacency matrix G:
[0 1 0 0 1 1 1 1 0 0 0 0 0 0 0 0]
[1 0 0 0 1 1 1 1 0 0 0 0 0 0 0 0]
[0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0]
[0 0 1 0 1 1 1 1 0 0 0 0 0 0 0 0]
[1 1 1 1 0 1 0 0 0 0 0 0 0 0 0 0]
[1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0]
[1 1 1 1 0 0 0 1 0 0 0 0 0 0 0 0]
[1 1 1 1 0 0 1 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 1 0 0 1 1 1 1]
[0 0 0 0 0 0 0 0 1 0 0 0 1 1 1 1]
[0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1]
[0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 1]
[0 0 0 0 0 0 0 0 1 1 1 1 0 1 0 0]
[0 0 0 0 0 0 0 0 1 1 1 1 1 0 0 0]
[0 0 0 0 0 0 0 0 1 1 1 1 0 0 0 1]
[0 0 0 0 0 0 0 0 1 1 1 1 0 0 1 0]
Complement of the primal graph adjacency matrix G':
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

NameError: name 'count_complete_subgraphs' is not defined

In [3]:





from sage.all import *
from itertools import combinations

def is_transversal(t, hypergraph):
    """Check if t is a transversal for the hypergraph."""
    return all(any(vertex in edge for vertex in t) for edge in hypergraph)

def is_minimal(t, transversals):
    """Check if t is minimal among the given transversals."""
    return not any(set(t) > set(other) for other in transversals)

def generate_transversal_hypergraph(incidence_matrix):
    vertices = range(len(incidence_matrix[0]))  # Assuming all vertices are represented in the incidence matrix
    hypergraph = [{i for i, val in enumerate(row) if val} for row in incidence_matrix]
    transversals = []

    for r in range(1, len(vertices) + 1):
        for combo in combinations(vertices, r):
            if is_transversal(combo, hypergraph) and is_minimal(combo, transversals):
                transversals.append(combo)

    # Convert transversals to incidence matrix format
    transversal_matrix = [[1 if i in t else 0 for i in vertices] for t in transversals]
    return Matrix(ZZ, transversal_matrix)

# Example usage:
incidence_matrix = Matrix(ZZ, [
    [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1]])

# Generate the transversal hypergraph incidence matrix
transversal_matrix = generate_transversal_hypergraph(incidence_matrix)

# Print the entire transversal hypergraph incidence matrix in list notation
print(transversal_matrix)


[1 0 1 0 1 0 1 0 1 0 1 0]
[1 0 1 0 1 0 1 0 0 1 0 1]
[1 0 1 0 0 1 0 1 1 0 1 0]
[1 0 1 0 0 1 0 1 0 1 0 1]
[0 1 0 1 1 0 1 0 1 0 1 0]
[0 1 0 1 1 0 1 0 0 1 0 1]
[0 1 0 1 0 1 0 1 1 0 1 0]
[0 1 0 1 0 1 0 1 0 1 0 1]


In [4]:
from sage.all import *
from itertools import combinations

def find_cycles(incidence_matrix):
    num_hyperedges = incidence_matrix.nrows()
    num_vertices = incidence_matrix.ncols()
    cycles = []

    # Iterate over all possible cycle lengths 3 and more
    for cycle_length in range(3, num_hyperedges + 1):
        for cycle_edges in combinations(range(num_hyperedges), cycle_length):
            # Check if it's a valid cycle (first and last edges must intersect)
            if not any(incidence_matrix[cycle_edges[0], v] == 1 and incidence_matrix[cycle_edges[-1], v] == 1 for v in range(num_vertices)):
                continue

            valid_cycle = True

            # Check if all consecutive edges in the cycle share at least one vertex
            for i in range(cycle_length - 1):
                if not any(incidence_matrix[cycle_edges[i], v] == 1 and incidence_matrix[cycle_edges[i + 1], v] == 1 for v in range(num_vertices)):
                    valid_cycle = False
                    break

            if valid_cycle:
                cycles.append(cycle_edges)

    return cycles

# Define the incidence matrix
incidence_matrix = Matrix([
    [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1]])

# Find and print all cycles
cycles = find_cycles(incidence_matrix)
if cycles:
    print("Cycles detected in the hypergraph:")
    for cycle in cycles:
        print(cycle)
else:
    print("There are no cycles.")


Cycles detected in the hypergraph:
(0, 1, 2, 3)
(4, 5, 6, 7)
(8, 9, 10, 11)


In [15]:
D4 = DihedralGroup(4)
C3 = CyclicPermutationGroup(3)

# Use GAP to compute the wreath product
gap_D4 = gap(D4)
gap_C3 = gap(C3)
wreath_product = gap_D4.WreathProduct(gap_C3)

# Get the generators and the structure in terms of direct and semidirect products
structure_description = wreath_product.StructureDescription()

print(structure_description)

(D8 x D8 x D8) : C3


In [ ]:
D4 = DihedralGroup(4)
C3 = CyclicPermutationGroup(3)
C2 = CyclicPermutationGroup(2)

gap_D4 = gap(D4)
gap_C3 = gap(C3)
gap_C2 = gap(C2)

wreath_product_1 = gap_D4.WreathProduct(gap_C3)
wreath_product_2 = wreath_product_1.WreathProduct(gap_C2)

structure_description = wreath_product_2.StructureDescription()
print(structure_description)

IOStream.flush timed out
IOStream.flush timed out
